In [ ]:
import json
import sys
from typing import List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from simulate_markov_utils import plot_markov_df
from sklearn.metrics import auc, roc_curve
import seaborn as sns

sys.path.append('..')
from multiple_monitors import prepare_multiple_monitors_data
from one_turn_debate import prepare_one_turn_debate_data
from plotting_utils import (
    load_in_combined_results,
    load_monitoring_data,
    plot_roc_curves,
    plot_suspiciousness_distributions,
)
from reflect_on_decision import prepare_reflect_on_decision_data
from turn_by_turn_monitoring import prepare_read_the_notes_data, prepare_turn_by_turn_data

from utils.analysis_utils import analyze_all_tasks, extract_verdict
from utils.path_utils import resolve_path

# Constants
MAIN_AGENT_MODELS = ['gpt-4o', 'claude-3-5-sonnet-20241022']
MONITOR_AGENT_MODELS = ['gpt-4o']
DEFAULT_TRANSCRIPT_TYPES = ["transcripts", "transcripts_no_hint", "transcripts_no_secondary_task"]
DOWNSAMPLE_FACTORS = [1, 2, 3, 4, 5, 10]
# Load combined results once at module level

In [ ]:
include_failed_attempts = False
include_failed_main_task = False
give_hint = False
all_dfs = []
combined_results = load_in_combined_results(include_failed_attempts, include_failed_main_task)
all_dfs.append(combined_results)
turn_by_turn_monitoring_df = prepare_turn_by_turn_data(base_dir='monitor_responses/turn_by_turn_monitoring_2', include_failed_attempts=include_failed_attempts, include_failed_main_task=include_failed_main_task, downsample_factors=DOWNSAMPLE_FACTORS, suffix='turn_by_turn')
all_dfs.append(turn_by_turn_monitoring_df)
read_the_notes_df = prepare_read_the_notes_data(base_dir='monitor_responses/read_the_notes', include_failed_attempts=include_failed_attempts, include_failed_main_task=include_failed_main_task)
# all_dfs.append(read_the_notes_df)
multiple_monitors_df = prepare_multiple_monitors_data(base_dir='monitor_responses/multiple_monitors_2', include_failed_attempts=include_failed_attempts, include_failed_main_task=include_failed_main_task)
all_dfs.append(multiple_monitors_df)
one_turn_debate_df = prepare_one_turn_debate_data(base_dir='monitor_responses/one_turn_debate')
# all_dfs.append(one_turn_debate_df)
running_scratchpad_df = prepare_turn_by_turn_data(base_dir='monitor_responses/running_scratchpad', include_failed_attempts=include_failed_attempts, include_failed_main_task=include_failed_main_task, downsample_factors=[1], suffix='running_scratchpad')
# all_dfs.append(running_scratchpad_df)
read_the_running_scratchpad_df = prepare_read_the_notes_data(base_dir='monitor_responses/read_the_running_scratchpad', include_failed_attempts=include_failed_attempts, include_failed_main_task=include_failed_main_task)
# all_dfs.append(read_the_running_scratchpad_df)
reflect_on_decision_df = prepare_reflect_on_decision_data(base_dir='monitor_responses/reflect_on_decision', include_failed_attempts=include_failed_attempts, include_failed_main_task=include_failed_main_task)
# all_dfs.append(reflect_on_decision_df)

In [ ]:
# Common keys for merging
merge_keys = ['task_name', 'transcript_type', 'main_agent', 'monitor_agent', 'file_name']

# Start with the first dataframe
merged_df = all_dfs[0]

# Merge with each subsequent dataframe
for i, df in enumerate(all_dfs[1:], 1):
    # Get columns to drop (columns that exist in both dataframes except merge keys)
    cols_to_drop = [col for col in df.columns if col in merged_df.columns and col not in merge_keys]
    
    # Merge the dataframes
    merged_df = pd.merge(
        merged_df, 
        df.drop(columns=cols_to_drop, errors='ignore'), 
        on=merge_keys,
        how='inner'
    )
    print(f"Merged dataframe {i} of {len(all_dfs)-1}. Shape: {merged_df.shape}")

# Display the merged dataframe info
print(f"Final merged dataframe shape: {merged_df.shape}")
print("Columns in merged dataframe:")
for col in merged_df.columns:
    print(col)


In [ ]:
# metrics = ['final_suspiciousness_turn_by_turn'] + ['max_suspiciousness_1_turn_by_turn'] + ['final_suspiciousness_running_scratchpad'] + ['max_suspiciousness_1_running_scratchpad'] + ['average_suspiciousness_debate'] + ['max_suspiciousness_multiple_monitors'] + ['average_suspiciousness_reflection'] + ['max_suspiciousness_reflection']
metrics = ['final_suspiciousness_turn_by_turn', 'suspiciousness_1_multiple_monitors', 'suspiciousness_2_multiple_monitors', 'suspiciousness_3_multiple_monitors']
plot_roc_curves(merged_df, combined_results, give_hint=True, include_failed_attempts=include_failed_attempts, include_failed_main_task=include_failed_main_task, monitors=['gpt-4o'], metrics=metrics)

In [ ]:
merged_df

In [ ]:
plot_suspiciousness_distributions(merged_df, monitors=['gpt-4o'], metrics=metrics)

In [ ]:
merged_df.sort_values(by='final_suspiciousness_turn_by_turn', ascending=False)[merge_keys + metrics]

In [ ]:
# Fix the syntax error in np.corrcoef call
np.corrcoef([merged_df['final_suspiciousness_turn_by_turn'], merged_df['suspiciousness_1_multiple_monitors']])

In [ ]:
# Calculate correlation matrix for all numeric columns
correlation_matrix = merged_df.select_dtypes(include=['float64', 'int64']).corr()
print("Correlation matrix shape:", correlation_matrix.shape)

In [ ]:
# Display correlation matrix for specific metrics of interest
metrics_correlation = merged_df[metrics].corr()
print("Metrics correlation matrix:")
print(metrics_correlation)


In [ ]:
# For visualizing the full correlation matrix (if needed)
# Select top N correlated features to make the plot readable
def plot_correlation_matrix(corr_matrix, top_n=20):
    # Get the absolute correlation values and sort
    corr_abs = corr_matrix.abs()
    
    # If there are more than top_n columns, select the top_n most correlated
    if len(corr_matrix.columns) > top_n:
        # Get the average correlation for each column
        avg_corr = corr_abs.mean()
        top_cols = avg_corr.nlargest(top_n).index
        corr_subset = corr_matrix.loc[top_cols, top_cols]
    else:
        corr_subset = corr_matrix
    
    plt.figure(figsize=(16, 14))  # Larger figure size
    heatmap = sns.heatmap(corr_subset, annot=True, cmap='coolwarm', vmin=-1, vmax=1, 
                         annot_kws={"size": 22}, fmt=".2f")  # Much larger annotation font size
    
    # Rotate x-axis labels and make them larger
    plt.xticks(fontsize=18, rotation=45, ha='right')  # Significantly larger x-axis labels
    plt.yticks(fontsize=18, rotation=0, va="top")  # Significantly larger y-axis labels
    
    plt.title('Correlation Matrix of Top Features', fontsize=22, pad=20)  # Larger title with padding
    plt.tight_layout()
    plt.show()

# Uncomment to use this function
plot_correlation_matrix(metrics_correlation)


In [ ]:
turn_by_turn_monitoring_df.iloc[0].data